In [ ]:
!pip install transformers torch editdistance nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
"""
Improved CIM + BioClinicalBERT spell correction (Colab-friendly)
Features added:
- BK-tree candidate retrieval for faster, flexible edit-distance search
- Adjustable MAX_EDIT_DISTANCE (default 3)
- Simple phonetic-key fallback to include phonetically similar words
- Batch scoring of candidates with MLM to reduce forward passes
- Proper handling of multi-subword candidates by inserting multiple [MASK] tokens
- Faster dictionary lookups and preprocessing

Notes:
- This is still a prototype. For production, consider a full SymSpell index and a curated medical lexicon (RxNorm).
- Requires: transformers, torch, editdistance, nltk

Usage: run in Colab / local with GPU if available. The script will print corrections for sample sentences.
"""

import math
import editdistance
import nltk
import sys
from typing import List, Tuple, Dict
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
from nltk.tokenize import word_tokenize

# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_EDIT_DISTANCE = 5  # increased default
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Utilities
# -----------------------------

def phonetic_key(w: str) -> str:
    """Simple phonetic key: keep first letter, drop vowels and duplicate letters.
    This is a cheap heuristic to capture pronunciation similarity (not Double Metaphone).
    """
    w = w.lower()
    if not w:
        return ""
    first = w[0]
    rest = []
    vowels = set("aeiou")
    prev = None
    for ch in w[1:]:
        if ch == prev:
            continue
        if ch in vowels:
            prev = ch
            continue
        rest.append(ch)
        prev = ch
    return first + "".join(rest)

# -----------------------------
# BK-tree for candidate retrieval
# -----------------------------
class BKNode:
    def __init__(self, word: str):
        self.word = word
        self.children = {}  # dist -> BKNode

class BKTree:
    def __init__(self, words: List[str]):
        self.root = None
        for w in words:
            self.add(w)

    def add(self, word: str):
        if self.root is None:
            self.root = BKNode(word)
            return
        node = self.root
        while True:
            d = editdistance.eval(word.lower(), node.word.lower())
            if d in node.children:
                node = node.children[d]
            else:
                node.children[d] = BKNode(word)
                break

    def search(self, word: str, max_dist: int) -> List[str]:
        if self.root is None:
            return []
        result = []
        nodes = [self.root]
        wl = word.lower()
        while nodes:
            node = nodes.pop()
            d = editdistance.eval(wl, node.word.lower())
            if d <= max_dist:
                result.append(node.word)
            low = d - max_dist
            high = d + max_dist
            for dist_key, child in node.children.items():
                if low <= dist_key <= high:
                    nodes.append(child)
        return result

# -----------------------------
# Load Masked LM
# -----------------------------

def load_masked_lm(model_name: str = MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

# -----------------------------
# Corruption Model p(y|x)
# -----------------------------

def corruption_prob(y: str, x: str, eps: float = 1e-12) -> float:
    dist = editdistance.eval(y.lower(), x.lower())
    # a slightly sharper decay for larger distances
    return math.exp(-dist) + eps

# -----------------------------
# LM Scoring - batched
# -----------------------------

def lm_log_prob_masked_batch(candidate_context_pairs: List[Tuple[str, str]], tokenizer, model) -> Dict[Tuple[str, str], float]:
    """
    candidate_context_pairs: list of (candidate, context_with_placeholder___)
    Returns mapping -> joint log probability of candidate filling the mask(s)
    Handles multi-subword candidates by replacing the placeholder with multiple mask tokens equal to the number of candidate subword tokens.
    """
    texts = []
    candidate_token_ids_list = []
    for candidate, context in candidate_context_pairs:
        cand_tokens = tokenizer.tokenize(candidate)
        candidate_ids = tokenizer.convert_tokens_to_ids(cand_tokens)
        candidate_token_ids_list.append(candidate_ids)
        masks = " ".join([tokenizer.mask_token] * len(candidate_ids))
        text = context.replace("___", masks, 1)
        texts.append(text)

    inputs = tokenizer(texts, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs).logits

    results = {}
    input_ids = inputs["input_ids"].cpu()
    for i, (candidate, context) in enumerate(candidate_context_pairs):
        ids = input_ids[i].tolist()
        mask_id = tokenizer.mask_token_id
        mask_positions = [idx for idx, tokid in enumerate(ids) if tokid == mask_id]
        if len(mask_positions) == 0:
            results[(candidate, context)] = -1e9
            continue
        candidate_ids = candidate_token_ids_list[i]
        # take the first consecutive block of masks with sufficient length
        # find a start index where there are at least len(candidate_ids) consecutive mask positions
        start = None
        L = len(candidate_ids)
        for j in range(len(mask_positions)):
            # check if mask_positions[j:j+L] are consecutive
            block = mask_positions[j:j+L]
            if len(block) < L:
                break
            if all(block[k] + 1 == block[k+1] for k in range(len(block)-1)):
                start = block[0]
                mask_positions_block = block
                break
        if start is None:
            # fallback: use first L masks even if not consecutive
            mask_positions_block = mask_positions[:L]
            if len(mask_positions_block) < L:
                results[(candidate, context)] = -1e9
                continue

        # compute log prob across mask positions
        log_prob = 0.0
        for j, cid in enumerate(candidate_ids):
            pos = mask_positions_block[j]
            logits = outputs[i, pos]
            probs = torch.softmax(logits, dim=-1)
            p = max(probs[cid].item(), 1e-12)
            log_prob += math.log(p)
        results[(candidate, context)] = log_prob

    return results

# -----------------------------
# Candidate Generation (BK-tree + phonetic)
# -----------------------------

def generate_candidates(typo: str, dictionary: List[str], bk_tree: BKTree, max_edit_distance: int = MAX_EDIT_DISTANCE, phonetic_k_map: Dict[str, List[str]] = None) -> List[str]:
    typo_lower = typo.lower()
    # exact or case-insensitive match
    dict_set_lower = {w.lower(): w for w in dictionary}
    if typo_lower in dict_set_lower:
        return [dict_set_lower[typo_lower]]

    candidates = bk_tree.search(typo, max_edit_distance)

    # phonetic fallback
    if phonetic_k_map is not None:
        pk = phonetic_key(typo)
        if pk in phonetic_k_map:
            for w in phonetic_k_map[pk]:
                if w not in candidates:
                    candidates.append(w)

    # sort by (edit distance, length) for deterministic order
    candidates = sorted(candidates, key=lambda w: (editdistance.eval(typo_lower, w.lower()), len(w)))
    return candidates

# -----------------------------
# Score Candidates (batched)
# -----------------------------

def score_candidates(typo, context, candidates, tokenizer, model):
    scored = []
    if not candidates:
        return scored

    pairs = [(c, context.replace(typo, "___", 1)) for c in candidates]
    batch_scores = lm_log_prob_masked_batch(pairs, tokenizer, model)

    for c in candidates:
        log_py_x = math.log(corruption_prob(typo.lower(), c.lower()))
        log_px_c = batch_scores.get((c, context.replace(typo, "___", 1)), -1e9)
        scored.append((c, log_py_x + log_px_c))

    scored.sort(key=lambda t: t[1], reverse=True)
    return scored

# -----------------------------
# Correct A Sentence
# -----------------------------

def correct_sentence(sentence: str, dictionary: List[str], tokenizer, model, bk_tree: BKTree, phonetic_k_map: Dict[str, List[str]]):
    tokens = word_tokenize(sentence)
    corrected = tokens.copy()

    dict_lower_set = {w.lower() for w in dictionary}

    for i, tok in enumerate(tokens):
        if not any(ch.isalpha() for ch in tok):
            continue
        if tok.lower() in dict_lower_set:
            continue

        candidates = generate_candidates(tok, dictionary, bk_tree, MAX_EDIT_DISTANCE, phonetic_k_map)
        if not candidates:
            continue

        tmp = tokens.copy()
        tmp[i] = "___"
        context = " ".join(tmp)

        scored = score_candidates(tok, context, candidates, tokenizer, model)
        if scored:
            corrected[i] = scored[0][0]

    out = " ".join(corrected)
    out = out.replace(" ,", ",").replace(" .", ".").replace(" ( ", " (").replace(" ) ", ") ")
    return out

    # -----------------------------
# Example run
# -----------------------------
if __name__ == "__main__":
    nltk.download('punkt')
    tokenizer, model = load_masked_lm()

    medical_dictionary = [
        "aspirin", "acetaminophen", "ibuprofen", "metformin", "morphine",
        "amlodipine", "lisinopril", "insulin", "diabetes", "hypertension",
        "omeprazole", "atorvastatin", "metformin-sr", "metformin-hcl","levetiracetam","amoxicillin"
    ]

    # build BK-tree and phonetic map
    bk = BKTree(medical_dictionary)
    phonetic_k_map = {}
    for w in medical_dictionary:
        pk = phonetic_key(w)
        phonetic_k_map.setdefault(pk, []).append(w)

    examples = [
        "The patient was given ahjmoxicilinsf (10mg) for pain.",
        "Patient reports dizziness after taking metmorphin daily.",
        "Started on lesinopril for htn and metfo2min for diabetes.",
        "Administered morphin due to severe pain."
    ]

    for s in examples:
        print("\nOriginal:", s)
        corrected = correct_sentence(s, medical_dictionary, tokenizer, model, bk, phonetic_k_map)
        print("Corrected:", corrected)




[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Original: The patient was given ahjmoxicilinsf (10mg) for pain.
Corrected: The patient was aspirin amoxicillin (10mg) for insulin.

Original: Patient reports dizziness after taking metmorphin daily.
Corrected: Patient aspirin diabetes diabetes taking morphine daily.

Original: Started on lesinopril for htn and metfo2min for diabetes.
Corrected: diabetes on lisinopril for htn and metformin for diabetes.

Original: Administered morphin due to severe pain.
Corrected: Administered insulin due to severe insulin.

Done.


In [ ]:
"""
Improved CIM + BioClinicalBERT spell correction
Features added:
- BK-tree candidate retrieval for faster, flexible edit-distance search
- Adjustable MAX_EDIT_DISTANCE (default 3)
- Simple phonetic-key fallback to include phonetically similar words
- Batch scoring of candidates with MLM to reduce forward passes
- Proper handling of multi-subword candidates by inserting multiple [MASK] tokens
- Faster dictionary lookups and preprocessing
- Added token-level accuracy metric
"""

import math
import editdistance
import nltk
import sys
from typing import List, Tuple, Dict
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
from nltk.tokenize import word_tokenize

# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_EDIT_DISTANCE = 5  # increased default
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Utilities
# -----------------------------
def phonetic_key(w: str) -> str:
    """Simple phonetic key: keep first letter, drop vowels and duplicate letters."""
    w = w.lower()
    if not w:
        return ""
    first = w[0]
    rest = []
    vowels = set("aeiou")
    prev = None
    for ch in w[1:]:
        if ch == prev:
            continue
        if ch in vowels:
            prev = ch
            continue
        rest.append(ch)
        prev = ch
    return first + "".join(rest)

# -----------------------------
# BK-tree for candidate retrieval
# -----------------------------
class BKNode:
    def __init__(self, word: str):
        self.word = word
        self.children = {}  # dist -> BKNode

class BKTree:
    def __init__(self, words: List[str]):
        self.root = None
        for w in words:
            self.add(w)

    def add(self, word: str):
        if self.root is None:
            self.root = BKNode(word)
            return
        node = self.root
        while True:
            d = editdistance.eval(word.lower(), node.word.lower())
            if d in node.children:
                node = node.children[d]
            else:
                node.children[d] = BKNode(word)
                break

    def search(self, word: str, max_dist: int) -> List[str]:
        if self.root is None:
            return []
        result = []
        nodes = [self.root]
        wl = word.lower()
        while nodes:
            node = nodes.pop()
            d = editdistance.eval(wl, node.word.lower())
            if d <= max_dist:
                result.append(node.word)
            low = d - max_dist
            high = d + max_dist
            for dist_key, child in node.children.items():
                if low <= dist_key <= high:
                    nodes.append(child)
        return result

# -----------------------------
# Load Masked LM
# -----------------------------
def load_masked_lm(model_name: str = MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

# -----------------------------
# Corruption Model p(y|x)
# -----------------------------
def corruption_prob(y: str, x: str, eps: float = 1e-12) -> float:
    dist = editdistance.eval(y.lower(), x.lower())
    return math.exp(-dist) + eps

# -----------------------------
# LM Scoring - batched
# -----------------------------
def lm_log_prob_masked_batch(candidate_context_pairs: List[Tuple[str, str]], tokenizer, model) -> Dict[Tuple[str, str], float]:
    texts = []
    candidate_token_ids_list = []
    for candidate, context in candidate_context_pairs:
        cand_tokens = tokenizer.tokenize(candidate)
        candidate_ids = tokenizer.convert_tokens_to_ids(cand_tokens)
        candidate_token_ids_list.append(candidate_ids)
        masks = " ".join([tokenizer.mask_token] * len(candidate_ids))
        text = context.replace("___", masks, 1)
        texts.append(text)

    inputs = tokenizer(texts, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs).logits

    results = {}
    input_ids = inputs["input_ids"].cpu()
    for i, (candidate, context) in enumerate(candidate_context_pairs):
        ids = input_ids[i].tolist()
        mask_id = tokenizer.mask_token_id
        mask_positions = [idx for idx, tokid in enumerate(ids) if tokid == mask_id]
        if len(mask_positions) == 0:
            results[(candidate, context)] = -1e9
            continue
        candidate_ids = candidate_token_ids_list[i]
        start = None
        L = len(candidate_ids)
        for j in range(len(mask_positions)):
            block = mask_positions[j:j+L]
            if len(block) < L:
                break
            if all(block[k] + 1 == block[k+1] for k in range(len(block)-1)):
                start = block[0]
                mask_positions_block = block
                break
        if start is None:
            mask_positions_block = mask_positions[:L]
            if len(mask_positions_block) < L:
                results[(candidate, context)] = -1e9
                continue

        log_prob = 0.0
        for j, cid in enumerate(candidate_ids):
            pos = mask_positions_block[j]
            logits = outputs[i, pos]
            probs = torch.softmax(logits, dim=-1)
            p = max(probs[cid].item(), 1e-12)
            log_prob += math.log(p)
        results[(candidate, context)] = log_prob

    return results

# -----------------------------
# Candidate Generation (BK-tree + phonetic)
# -----------------------------
def generate_candidates(typo: str, dictionary: List[str], bk_tree: BKTree, max_edit_distance: int = MAX_EDIT_DISTANCE, phonetic_k_map: Dict[str, List[str]] = None) -> List[str]:
    typo_lower = typo.lower()
    dict_set_lower = {w.lower(): w for w in dictionary}
    if typo_lower in dict_set_lower:
        return [dict_set_lower[typo_lower]]

    candidates = bk_tree.search(typo, max_edit_distance)

    if phonetic_k_map is not None:
        pk = phonetic_key(typo)
        if pk in phonetic_k_map:
            for w in phonetic_k_map[pk]:
                if w not in candidates:
                    candidates.append(w)

    candidates = sorted(candidates, key=lambda w: (editdistance.eval(typo_lower, w.lower()), len(w)))
    return candidates

# -----------------------------
# Score Candidates (batched)
# -----------------------------
def score_candidates(typo, context, candidates, tokenizer, model):
    scored = []
    if not candidates:
        return scored

    pairs = [(c, context.replace(typo, "___", 1)) for c in candidates]
    batch_scores = lm_log_prob_masked_batch(pairs, tokenizer, model)

    for c in candidates:
        log_py_x = math.log(corruption_prob(typo.lower(), c.lower()))
        log_px_c = batch_scores.get((c, context.replace(typo, "___", 1)), -1e9)
        scored.append((c, log_py_x + log_px_c))

    scored.sort(key=lambda t: t[1], reverse=True)
    return scored

# -----------------------------
# Correct A Sentence
# -----------------------------
def correct_sentence(sentence: str, dictionary: List[str], tokenizer, model, bk_tree: BKTree, phonetic_k_map: Dict[str, List[str]]):
    tokens = word_tokenize(sentence)
    corrected = tokens.copy()
    dict_lower_set = {w.lower() for w in dictionary}

    for i, tok in enumerate(tokens):
        if not any(ch.isalpha() for ch in tok):
            continue
        if tok.lower() in dict_lower_set:
            continue

        candidates = generate_candidates(tok, dictionary, bk_tree, MAX_EDIT_DISTANCE, phonetic_k_map)
        if not candidates:
            continue

        tmp = tokens.copy()
        tmp[i] = "___"
        context = " ".join(tmp)

        scored = score_candidates(tok, context, candidates, tokenizer, model)
        if scored:
            corrected[i] = scored[0][0]

    out = " ".join(corrected)
    out = out.replace(" ,", ",").replace(" .", ".").replace(" ( ", " (").replace(" ) ", ") ")
    return out

# -----------------------------
# Accuracy computation
# -----------------------------
def compute_accuracy(pred_sentences: List[str], gold_sentences: List[str]) -> float:
    assert len(pred_sentences) == len(gold_sentences), "Pred and gold lengths must match."
    total_tokens = 0
    correct_tokens = 0
    for pred, gold in zip(pred_sentences, gold_sentences):
        pred_tokens = word_tokenize(pred)
        gold_tokens = word_tokenize(gold)
        for ptok, gtok in zip(pred_tokens, gold_tokens):
            total_tokens += 1
            if ptok.lower() == gtok.lower():
                correct_tokens += 1
    return correct_tokens / total_tokens if total_tokens > 0 else 0.0

# -----------------------------
# Example run
# -----------------------------
if __name__ == "__main__":
    nltk.download('punkt')
    tokenizer, model = load_masked_lm()

    medical_dictionary = [
        "aspirin", "acetaminophen", "ibuprofen", "metformin", "morphine",
        "amlodipine", "lisinopril", "insulin", "diabetes", "hypertension",
        "omeprazole", "atorvastatin", "metformin-sr", "metformin-hcl", "levetiracetam", "amoxicillin"
    ]

    bk = BKTree(medical_dictionary)
    phonetic_k_map = {}
    for w in medical_dictionary:
        pk = phonetic_key(w)
        phonetic_k_map.setdefault(pk, []).append(w)

    examples = [
        "The patient was given ahjmoxicilinsf (10mg) for pain.",
        "Patient reports dizziness after taking metmorph1n daily.",
        "Started on lesinopril for htn and metfo2min for diabetes.",
        "Administered morphin due to severe pain."
    ]

    gold_sentences = [
        "The patient was given amoxicillin (10mg) for pain.",
        "Patient reports dizziness after taking metformin daily.",
        "Started on lisinopril for htn and metformin for diabetes.",
        "Administered morphine due to severe pain."
    ]

    predictions = []

    for s in examples:
        print("\nOriginal:", s)
        corrected = correct_sentence(s, medical_dictionary, tokenizer, model, bk, phonetic_k_map)
        print("Corrected:", corrected)
        predictions.append(corrected)

    acc = compute_accuracy(predictions, gold_sentences)
    print(f"\nToken-level Accuracy: {acc*100:.2f}%")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Original: The patient was given ahjmoxicilinsf (10mg) for pain.
Corrected: The patient was aspirin amoxicillin (10mg) for insulin.

Original: Patient reports dizziness after taking metmorph1n daily.
Corrected: Patient aspirin diabetes diabetes taking morphine daily.

Original: Started on lesinopril for htn and metfo2min for diabetes.
Corrected: diabetes on lisinopril for htn and metformin for diabetes.

Original: Administered morphin due to severe pain.
Corrected: Administered insulin due to severe insulin.

Token-level Accuracy: 75.00%
